本 notebook 是 [M04.6 异常检测](../notes/06-anomaly-detection.qmd) 的实例讲解。

1. 1. Z-score、IQR 与遮蔽效应
2. 2. 从零 LOF 与框架对齐
3. 3. 三类新样本检测器与统一分数方向
4. 4. 阈值、基准率与漂移

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

In [1]:
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.spatial.distance import cdist
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. Z-score、IQR 与遮蔽效应

把少量极端值加入正态参考，比较均值标准差规则和分位规则；检查边缘正常不代表联合正常。

In [2]:
base = rng.normal(size=400)
observed = np.r_[base, np.repeat(12.0, 20)]
z = (observed - observed.mean()) / observed.std()
q1, q3 = np.quantile(observed, [0.25, 0.75])
iqr = q3 - q1
print(
    "Z>3 标记数:",
    np.sum(np.abs(z) > 3),
    "IQR 标记数:",
    np.sum((observed < q1 - 1.5 * iqr) | (observed > q3 + 1.5 * iqr)),
)
x = rng.multivariate_normal([0, 0], [[1, 0.99], [0.99, 1]], 1000)
point = np.array([1.0, -1.0])
cov = np.cov(x, rowvar=False)
print(
    "点的边缘绝对值:",
    np.abs(point),
    "联合 Mahalanobis 距离:",
    np.sqrt(point @ np.linalg.solve(cov, point)),
)

Z>3 标记数: 20 IQR 标记数: 22
点的边缘绝对值: [1. 1.] 联合 Mahalanobis 距离: 13.891580039614858


读法：极端值会反过来抬高标准差，IQR 相对稳健；相关变量中的异常方向可能无法由逐列阈值发现。阈值规则本身不是异常原因的判断。

## 2. 从零 LOF 与框架对齐

计算 k 距离、可达距离、局部可达密度与密度比，在无距离并列的合成点上核验训练点 LOF。

In [3]:
X = rng.normal(size=(60, 2))
k = 8
distance = cdist(X, X)
np.fill_diagonal(distance, np.inf)
neighbors = np.argsort(distance, axis=1)[:, :k]
k_distance = np.take_along_axis(distance, neighbors, axis=1).max(axis=1)
neighbor_distance = np.take_along_axis(distance, neighbors, axis=1)
reach = np.maximum(neighbor_distance, k_distance[neighbors])
lrd = 1 / (reach.mean(axis=1) + 1e-10)
lof = (lrd[neighbors] / lrd[:, None]).mean(axis=1)
reference = LocalOutlierFactor(n_neighbors=k).fit(X)
print("LOF 最大差:", np.max(np.abs(lof + reference.negative_outlier_factor_)))
assert np.allclose(lof, -reference.negative_outlier_factor_, atol=1e-10)

LOF 最大差: 4.440892098500626e-16


读法：LOF 比较一个点与邻居的局部密度，接近一表示相近，大于一表示更稀疏；本例只对齐训练点模式，不混用 novelty=True 的新样本评分。

## 3. 三类新样本检测器与统一分数方向

只用正常参考拟合 Isolation Forest、One-Class SVM 与 novelty LOF，再对独立混合样本输出异常分数和排名指标。

In [4]:
train = rng.normal(size=(600, 2))
test = np.r_[rng.normal(size=(1800, 2)), rng.normal(3, 1, (60, 2))]
y = np.r_[np.zeros(1800), np.ones(60)]
models = {
    "IF": IsolationForest(n_estimators=100, random_state=42),
    "OCSVM": make_pipeline(StandardScaler(), OneClassSVM(nu=0.05, gamma="scale")),
    "LOF": make_pipeline(StandardScaler(), LocalOutlierFactor(novelty=True, n_neighbors=25)),
}
for name, model in models.items():
    model.fit(train)
    score = -model.score_samples(test)
    print(name, "ROC AUC", roc_auc_score(y, score), "AP", average_precision_score(y, score))
print("AP 随机排序基准:", y.mean())

IF ROC AUC 0.9882962962962963 AP 0.8863858144352411
OCSVM ROC AUC 0.9377777777777778 AP 0.8644008714318288
LOF ROC AUC 0.9868055555555556 AP 0.8650198871383565
AP 随机排序基准: 0.03225806451612903


读法：统一规定分数越高越异常，避免符号反转；该例异常类型已知且合成，指标只支持这一机制，不能代表未知故障的召回率。

## 4. 阈值、基准率与漂移

用独立正常校准数据定分位阈值，比较同分布和均值漂移后的误报，再用 Bayes 公式计算低基准率下的告警精确率。

In [5]:
model = models["IF"]
calibration = rng.normal(size=(2000, 2))
threshold = np.quantile(-model.score_samples(calibration), 0.99)
for shift in [0, 0.6, 1.2]:
    nominal = rng.normal(shift, 1, (5000, 2))
    print("shift", shift, "正常样本告警率", np.mean(-model.score_samples(nominal) > threshold))
for prevalence in [0.001, 0.01, 0.1]:
    sensitivity = 0.9
    fpr = 0.01
    precision = sensitivity * prevalence / (sensitivity * prevalence + fpr * (1 - prevalence))
    print("异常率", prevalence, "固定灵敏度与误报率下的 precision", precision)

shift 0 正常样本告警率 0.0072
shift 0.6 正常样本告警率 0.0228
shift 1.2 正常样本告警率 0.1098
异常率 0.001 固定灵敏度与误报率下的 precision 0.08264462809917356
异常率 0.01 固定灵敏度与误报率下的 precision 0.4761904761904762
异常率 0.1 固定灵敏度与误报率下的 precision 0.9090909090909092


读法：分位阈值针对校准分布，不保证漂移后保持相同误报率；即使灵敏度和误报率固定，极低异常率也会让告警包含大量正常样本。

## 小结

异常检测需要参考、表示、分数与决策四层定义。方法的几何假设决定它偏好发现什么，校准数据和成本决定阈值，基准率决定告警的实际价值。把高分样本送去核查，再结合机制证据做处理，才能让异常检测服务科学数据质量。